In [10]:
import torch
from torchvision import datasets, transforms
import numpy as np
# Set the threshold to a high value so that the full array is printed.
np.set_printoptions(threshold=np.inf)

# Define the transformation: resize images to 32x32 and convert them to tensors.
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])

# Download and load the MNIST dataset (using the train set here)
mnist_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)

# Get a single image and its label (the first image)
image, label = mnist_dataset[0]

# image is a PyTorch tensor with shape [1, 32, 32]
print("Image tensor size:")
print(image.size())

# Convert the tensor to a NumPy array and print it
image_np = image.numpy()
print("Expanded image as NumPy array:")
print(image_np)



Image tensor size:
torch.Size([1, 32, 32])
Expanded image as NumPy array:
[[[0.         0.         0.         0.         0.         0.
   0.         0.         0.         0.         0.         0.
   0.         0.         0.         0.         0.         0.
   0.         0.         0.         0.         0.         0.
   0.         0.         0.         0.         0.         0.
   0.         0.        ]
  [0.         0.         0.         0.         0.         0.
   0.         0.         0.         0.         0.         0.
   0.         0.         0.         0.         0.         0.
   0.         0.         0.         0.         0.         0.
   0.         0.         0.         0.         0.         0.
   0.         0.        ]
  [0.         0.         0.         0.         0.         0.
   0.         0.         0.         0.         0.         0.
   0.         0.         0.         0.         0.         0.
   0.         0.         0.         0.         0.         0.
   0.         0.    

打印fixed number

In [11]:
# Quantization parameters for fixed-point representation:
# Total bits = 8, Fractional bits = 3 (scale factor = 2^3 = 8)
total_bits = 8
frac_bits = 3
scale = 2 ** frac_bits

# For an 8-bit signed number, the range is:
min_val = -(1 << (total_bits - 1))  # -128
max_val = (1 << (total_bits - 1)) - 1  # 127

# Quantize each pixel: multiply by scale, round, and clip to range.
quantized_int = np.clip(np.round(image_np * scale), min_val, max_val).astype(np.int8)

# Recover the fixed-point float representation:
quantized_float = quantized_int / scale

# Define formatting parameters for printing fixed-point floats.
total_width = 8
decimal_places = 3
formatter_float = {'float_kind': lambda x: f"{x:{total_width}.{decimal_places}f}"}
formatted_fixed = np.array2string(quantized_float, formatter=formatter_float)
print("\nQuantized fixed-point image (float representation):")
print(formatted_fixed)

# ------------------------------------------------------------------
# Functions to convert an 8-bit signed integer to binary and hexadecimal strings.
def int8_to_bin(x):
    # Convert a signed int8 to its two's complement unsigned representation.
    x_unsigned = x if x >= 0 else (1 << total_bits) + x
    return format(x_unsigned, f'0{total_bits}b')

def int8_to_hex(x):
    # Convert a signed int8 to its two's complement unsigned representation.
    x_unsigned = x if x >= 0 else (1 << total_bits) + x
    # For 8 bits, we want exactly 2 hex digits.
    return format(x_unsigned, '02x')

# Vectorize the conversion functions so they can be applied elementwise.
vec_int8_to_bin = np.vectorize(int8_to_bin)
vec_int8_to_hex = np.vectorize(int8_to_hex)

# Create arrays of binary and hexadecimal strings for the quantized image.
bin_array = vec_int8_to_bin(quantized_int)
hex_array = vec_int8_to_hex(quantized_int)

print("\nQuantized image in binary (8 bits):")
print(np.array2string(bin_array, separator=', '))

print("\nQuantized image in hexadecimal (2 hex digits):")
print(np.array2string(hex_array, separator=', '))


Quantized fixed-point image (float representation):
[[[   0.000    0.000    0.000    0.000    0.000    0.000    0.000
      0.000    0.000    0.000    0.000    0.000    0.000    0.000
      0.000    0.000    0.000    0.000    0.000    0.000    0.000
      0.000    0.000    0.000    0.000    0.000    0.000    0.000
      0.000    0.000    0.000    0.000]
  [   0.000    0.000    0.000    0.000    0.000    0.000    0.000
      0.000    0.000    0.000    0.000    0.000    0.000    0.000
      0.000    0.000    0.000    0.000    0.000    0.000    0.000
      0.000    0.000    0.000    0.000    0.000    0.000    0.000
      0.000    0.000    0.000    0.000]
  [   0.000    0.000    0.000    0.000    0.000    0.000    0.000
      0.000    0.000    0.000    0.000    0.000    0.000    0.000
      0.000    0.000    0.000    0.000    0.000    0.000    0.000
      0.000    0.000    0.000    0.000    0.000    0.000    0.000
      0.000    0.000    0.000    0.000]
  [   0.000    0.000    0.000    0.